# Toroidal Propeller BEMT Analysis

**Blade Element Momentum Theory (BEMT) solver — runs natively on Google Colab**

This notebook:
1. Clones the repository and installs dependencies
2. Runs the BEMT solver for all 4 propeller configurations
3. Compares predictions against experimental data from the paper
4. Generates publication-quality comparison plots

No OpenFOAM installation required for this notebook.

---
**Paper:** Palileo, Sabanal, Delicano & Gregorio — *Performance Analysis of Toroidal Blade Propellers for Small Fixed-Wing UAVs*

In [ ]:
import os, sys

REPO_URL = 'https://github.com/rp3gregorio/Propeller-CFD.git'
BRANCH   = 'claude/add-cfd-propeller-simulation-EEQqr'

# ── Detect environment ──────────────────────────────────────────────────
ON_COLAB = 'google.colab' in sys.modules or os.path.isdir('/content')

if ON_COLAB:
    REPO_DIR = '/content/Propeller-CFD'
    if not os.path.isdir(REPO_DIR):
        !git clone --branch {BRANCH} --depth 1 {REPO_URL} {REPO_DIR}
    else:
        !git -C {REPO_DIR} fetch origin
        !git -C {REPO_DIR} checkout {BRANCH}
        !git -C {REPO_DIR} pull origin {BRANCH}
else:
    # Running locally — find the repo root by looking for bemt/ directory
    _here = os.path.abspath(os.getcwd())
    for _candidate in [_here, os.path.dirname(_here), os.path.dirname(os.path.dirname(_here))]:
        if os.path.isdir(os.path.join(_candidate, 'bemt')):
            REPO_DIR = _candidate
            break
    else:
        raise RuntimeError(
            "Could not find the repo root (directory containing bemt/).\n"
            "Open Jupyter from inside the cloned Propeller-CFD folder."
        )
    print(f'Running locally. Repo root: {REPO_DIR}')

# Clear any cached Python imports so updated code is always used
for _mod in [m for m in sys.modules if m.startswith(('bemt', 'geometry', 'postprocessing'))]:
    del sys.modules[_mod]

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

import subprocess as _sp
print('Working directory:', os.getcwd())
print('Branch:', _sp.run('git branch --show-current', shell=True, capture_output=True, text=True, cwd=REPO_DIR).stdout.strip())
print('Latest commit:', _sp.run('git log --oneline -1', shell=True, capture_output=True, text=True, cwd=REPO_DIR).stdout.strip())

In [ ]:
# ── Step 2: Install Python dependencies ────────────────────────────────
!pip install -q numpy matplotlib pandas scipy

In [ ]:
# ── Step 3: Import BEMT solver and configs ─────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

from bemt.bemt_solver import BEMTSolver
from bemt.propeller_configs import CONFIGS, CONFIG_ORDER

print('Propeller configurations loaded:')
for name in CONFIG_ORDER:
    cfg = CONFIGS[name]
    solver = BEMTSolver(cfg, n_sections=60)
    print(f"  {name:30s}  span={solver.span*1000:.1f}mm  "
          f"σ={solver.n_blades * solver.blade_area / (np.pi * solver.r_tip**2):.3f}  "
          f"tip_loss={cfg['tip_loss']}")

In [ ]:
# ── Step 4: Run BEMT sweep — static thrust ─────────────────────────────
RPM_RANGE  = list(range(1000, 6001, 250))
WIND_SPEEDS = [0.0, 3.0, 9.0, 15.0]

bemt_results = {}
for name in CONFIG_ORDER:
    cfg    = CONFIGS[name]
    solver = BEMTSolver(cfg, n_sections=60)
    bemt_results[name] = {}
    for v in WIND_SPEEDS:
        res = solver.sweep_rpm(RPM_RANGE, V_inf=v)
        bemt_results[name][v] = res

print('BEMT sweep complete for all configurations and wind speeds.')

In [ ]:
# ── Step 5: Load experimental data ─────────────────────────────────────
exp_df = pd.read_csv('experimental_data/all_experimental.csv')
print(f'Experimental data: {len(exp_df)} data points')
print(exp_df.groupby(['config', 'v_inf_ms'])['rpm'].count().to_string())

In [ ]:
# ── Step 6: Plot Thrust vs RPM — all wind speeds ────────────────────────
def get_style(name):
    cfg = CONFIGS[name]
    return dict(color=cfg['color'], linestyle=cfg['linestyle'],
                marker=cfg['marker'], label=cfg['label'],
                markevery=3, markersize=5, linewidth=1.8)

titles = {0.0: 'Static Thrust (V∞ = 0)',
          3.0: 'Wind Tunnel V∞ = 3 m/s',
          9.0: 'Wind Tunnel V∞ = 9 m/s',
          15.0: 'Wind Tunnel V∞ = 15 m/s'}

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle('Thrust vs RPM: BEMT Predictions vs Experimental Data', fontsize=14, y=0.98)

for ax, v in zip(axes.flat, WIND_SPEEDS):
    # BEMT lines
    for name in CONFIG_ORDER:
        res = bemt_results[name][v]
        thrust_gf = np.array(res['T']) / 9.81 * 1000
        ax.plot(res['rpm'], thrust_gf, **get_style(name))

    # Experimental scatter
    subset = exp_df[np.isclose(exp_df['v_inf_ms'], v, atol=0.5)]
    for name in CONFIG_ORDER:
        cfg = CONFIGS[name]
        rows = subset[subset['config'] == name]
        if not rows.empty:
            ax.scatter(rows['rpm'], rows['T_gf'],
                       color=cfg['color'], marker='o', s=30, zorder=5,
                       facecolors='none', linewidths=1.5)

    ax.set_title(titles[v])
    ax.set_xlabel('RPM')
    ax.set_ylabel('Thrust [gf]')
    ax.grid(True, alpha=0.3)
    ax.set_xlim(1000, 6000)
    ax.set_ylim(bottom=0)

# Shared legend
handles, labels = axes[0, 0].get_legend_handles_labels()
for ax in axes.flat:
    if ax.get_legend(): ax.get_legend().remove()
fig.legend(handles, labels, loc='lower center', ncol=4, fontsize=9,
           bbox_to_anchor=(0.5, 0.01))

plt.tight_layout(rect=[0, 0.07, 1, 0.96])
os.makedirs('results/plots', exist_ok=True)
plt.savefig('results/plots/thrust_vs_rpm.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/plots/thrust_vs_rpm.png')

In [ ]:
# ── Step 7: CT vs J and Efficiency vs J ───────────────────────────────
RPM_REF = 4000
J_range = np.linspace(0.01, 0.80, 50)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle(f'Propeller Performance Polar at RPM = {RPM_REF}', fontsize=13)

for name in CONFIG_ORDER:
    cfg    = CONFIGS[name]
    solver = BEMTSolver(cfg, n_sections=60)
    res    = solver.sweep_advance(RPM_REF, J_range)
    sty    = get_style(name)
    ax1.plot(res['J'], res['CT'], **sty)
    ax2.plot(res['J'], np.clip(res['eta'], 0, 1), **sty)

ax1.set_xlabel('Advance ratio J')
ax1.set_ylabel('Thrust coefficient CT')
ax1.set_title('CT vs J')
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3)

ax2.set_xlabel('Advance ratio J')
ax2.set_ylabel('Propulsive efficiency η')
ax2.set_title('Efficiency vs J')
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)
ax2.set_ylim(0, 1)

plt.tight_layout()
plt.savefig('results/plots/performance_polar.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/plots/performance_polar.png')

In [ ]:
# ── Step 8: Thrust enhancement ratio (toroidal / conventional) ─────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Toroidal vs Conventional Thrust Ratio  (>1 = toroidal wins)', fontsize=13)

toroidal_configs = ['toroidal_low_gap', 'toroidal_medium_gap', 'toroidal_high_gap']

for ax, v in [(axes[0], 0.0), (axes[1], 15.0)]:
    T_conv = np.array(bemt_results['conventional'][v]['T'])
    for name in toroidal_configs:
        T_tor = np.array(bemt_results[name][v]['T'])
        ratio = T_tor / np.maximum(T_conv, 1e-6)
        cfg = CONFIGS[name]
        ax.plot(RPM_RANGE, ratio, color=cfg['color'], linestyle=cfg['linestyle'],
                label=cfg['label'], linewidth=1.8)

    ax.axhline(1.0, color='black', linewidth=1.0, linestyle='-', alpha=0.5, label='Conventional baseline')
    title = 'Static Thrust' if v == 0 else f'V∞ = {v:.0f} m/s'
    ax.set_title(title)
    ax.set_xlabel('RPM')
    ax.set_ylabel('Thrust ratio (toroidal / conventional)')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0.5, 1.5)
    ax.set_xlim(1000, 6000)

plt.tight_layout()
plt.savefig('results/plots/thrust_ratio.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Step 9: Summary table at 6000 RPM ─────────────────────────────────
print(f'\n{"="*65}')
print(f'  BEMT Summary at 6000 RPM')
print(f'{"="*65}')
print(f'{"Config":30s}  {"T_static":>10s}  {"T@15m/s":>10s}  {"CT":>8s}  {"η@J=0.2":>8s}')
print(f'{"─"*65}')

RPM_TARGET = 6000
for name in CONFIG_ORDER:
    cfg    = CONFIGS[name]
    solver = BEMTSolver(cfg, n_sections=60)
    r0     = solver.solve(RPM_TARGET, 0.0)
    r15    = solver.solve(RPM_TARGET, 15.0)
    r_j02  = solver.solve(RPM_TARGET, 0.2 * (RPM_TARGET/60) * 2*solver.r_tip)
    print(f'{cfg["label"]:30s}  {r0["T"]/9.81*1000:>8.1f}gf  '
          f'{r15["T"]/9.81*1000:>8.1f}gf  {r0["CT"]:>8.4f}  {r_j02["eta"]:>8.3f}')

print(f'\nNote: η@J=0.2 is propulsive efficiency at advance ratio J=0.2')

## Interpretation

Key findings from the BEMT analysis, consistent with the paper:

1. **Toroidal medium-gap (1.5")** produces the highest thrust at high RPM due to the optimal balance between blade area, aspect ratio, and tip-loss elimination.

2. **Tip-loss elimination** is the dominant advantage of the toroidal design. By connecting the blade tips, the bound vortex system is closed and tip vortices are suppressed.

3. **At low RPM**, the conventional propeller is competitive due to lower profile drag from the smaller, more slender blade geometry.

4. **At high wind speeds (15 m/s)**, toroidal advantage grows as the large blade area maintains attached flow at higher advance ratios.

5. **Efficiency trade-off**: The conventional propeller has marginally higher efficiency at cruise conditions (low J) due to lower profile drag, while toroidal designs excel in high-thrust applications.

---
Next: See `02_OpenFOAM_Setup.ipynb` for full 3D CFD case generation and execution.